# HIXL 资源管理

Initialize 与 Finalize 是 HIXL 创建与释放 engine 实例的核心接口。Initialize 创建 engine 实例并初始化内部资源池，使实例进入可用状态；Finalize 反向释放资源，结束实例生命周期。

理解这两个接口的关键：它们管理的是 engine 实例的生命周期边界，所有后续操作都在这个边界内执行。Initialize 是向系统申请通信资源的入口，Finalize 是归还资源的出口。

本节学习大纲如下：

- Initialize/Finalize 的概念与时机
- Initialize 使用方法
- Finalize 使用方法
- 开发注意事项
- 代码示例

## 1. Initialize/Finalize 的概念与时机

### 1.1 HIXL 实例的概念

HIXL engine 是一个通信端点，持有注册内存、链路和传输等资源。创建 `Hixl` 对象只是构造一个空实例，没有任何通信能力；Initialize 才是向系统申请通信资源、让 engine 进入可用状态的关键步骤。

这两个操作的职责边界：

- **创建 Hixl 对象**：构造空的 C++ 实例，不涉及资源分配
- **Initialize**：解析地址、创建资源池、启动监听（若为 Server），使 engine 进入可用状态

### 1.2 Initialize 的职责

Initialize 定义了 engine 实例的入口，它完成以下工作：

- 解析 local_engine，创建内部资源池
- 根据 options 配置通信模式（中转、Fabric Mem、AutoConnect 等）
- 若 local_engine 带端口，启动监听等待远端连接
- 返回 SUCCESS 表示 engine 已进入可用状态

Initialize 成功后，才能进行后续的内存注册、建链和传输操作。

### 1.3 Server/Client 角色

Server 和 Client 的区分来自 Initialize 时 local_engine 的写法：

<img src="./images/server_client_roles.png" alt="Server/Client角色示意图" width="80%">

| 角色 | local_engine 格式 | Initialize 行为 |
| --- | --- | --- |
| Server | `host_ip:port` | 启动端口监听，等待远端 Connect |
| Client | `host_ip` | 不启动监听，主动 Connect 到远端 |

角色区分的本质是"谁监听、谁主动连接"。Server 负责监听等待，Client 负责主动发起连接。

### 1.4 Finalize 的职责

- 停止监听（若有）
- 释放 engine 内部持有的通信资源
- 结束 engine 实例生命周期

> **注意**：Finalize 不负责释放用户显式分配的资源（如 `aclrtMalloc` 的内存），这些需要用户在 Finalize 前自行清理。

### 1.4 生命周期边界：成对使用的原因

HIXL 持有的通信资源（监听端口、RDMA/HCCS 硬件通道）不随进程自动创建或释放。Initialize 是向系统申请这些资源的入口，Finalize 是归还资源的出口。

Initialize 成功后，任何退出路径都必须调用 Finalize，否则资源释放顺序可能出问题。

## 2. Initialize 使用方法

### 2.1 接口签名

```cpp
Status Initialize(const AscendString &local_engine,
                  const std::map<AscendString, AscendString> &options);
```

### 2.2 local_engine 参数

`local_engine` 是一个地址字符串，用于唯一标识当前 engine 的通信端点，格式如下：

- **Server 写法**：`host_ip:port`，如 `10.10.10.1:26000`，启动端口监听等待远端 Connect
- **Client 写法**：`host_ip`，如 `10.10.10.2`，不启动监听，需主动 Connect 到远端

示例：

```
// Rank 0，Server engine 初始化，监听端口 26000
Initialize("10.10.10.1:26000", options)

// Rank 1，Client engine，不监听，需主动 Connect
Initialize("10.10.10.2", options)
```

**常见错误**：

- `10.10.10.1::26000`：IPv6 格式混用 IPv4 地址，解析失败
- `10.10.10.1:0`：端口 0 不启动监听，Server 角色失效
- 重复 local_engine：端点标识冲突，通信协议异常

### 2.3 options 参数

`options` 用于配置 engine 的通信行为。学习阶段建议先掌握以下两个常用项：

- `OPTION_AUTO_CONNECT`：设为 `"1"` 时自动建链，省去手动 Connect 调用
- `OPTION_BUFFER_POOL`：使用中转模式时配置 buffer 池大小

其他更多参数定义见 [HIXL接口.md](https://gitcode.com/cann/hixl/blob/master/docs/cpp/HIXL接口.md)。

### 2.4 返回值处理

Initialize 成功返回 `SUCCESS`，失败时常见错误码：

- `PARAM_INVALID`：参数错误，检查 local_engine 格式和 options 配置
- 其他错误：检查设备上下文是否正确设置

Initialize 失败时不能调用后续接口，需要先排查问题。

## 3. Finalize 使用方法

### 3.1 接口签名

```cpp
void Finalize();
```

### 3.2 释放时机与顺序

Finalize 必须在 Initialize 成功后调用，且需要在传输、链路、内存都完成清理后调用，这些接口功能将在后面章节依次介绍，当前只需有个印象即可。

释放顺序由依赖关系决定：

```
Transfer 依赖 Connect 创建的 Link
Link 依赖 RegisterMem 注册的 Memory
Memory 依赖 Initialize 创建的 Engine
```

因此反向释放顺序必须是：

`Transfer 完成 → Disconnect → DeregisterMem → aclrtFree → Finalize`


## 4. 开发注意事项

### 4.1 Server/Client 启动时序

Client Connect 前必须确保 Server 已 Initialize 并启动监听，时序颠倒会导致 Client 收到 TIMEOUT。

建议启动顺序：Server 先启动 → 等待监听就绪 → Client 再启动。

### 4.2 Server/Client 退出时序

Server Finalize 前必须确保所有 Client 已完成断链和远端读写操作：

- Server 提前 Finalize → Client 断链和传输过程报错
- Client 正在进行远端读写 → Server Finalize 导致 Client 访问失败

建议退出顺序：Client 先完成断链和远端读写 → Server 再调用 Finalize。

### 4.3 释放顺序错误的影响

| 错误操作 | 后果 |
| --- | --- |
| Finalize 前未 Disconnect | 链路资源未释放，远端可能收到异常 |
| Server 提前 Finalize | Client 的远端读写请求失败 |
| DeregisterMem 前 aclrtFree | 解注册时访问非法地址 |

### 4.4 多线程 context 管理

Initialize/Finalize 要求在 Initialize 同线程调用，或通过 `aclrtGetCurrentContext` / `aclrtSetCurrentContext` 切换 context。跨线程调用会导致接口返回错误。


## 5. 代码示例

下面代码示例展示 Initialize 与 Finalize 的最小调用流程：在成功调用 Initialize 后，engine 进入可用状态，随后在该生命周期边界内执行内存注册、建链、数据传输等操作；完成所有操作并清理相关资源后调用 Finalize 结束实例生命周期。后续章节将在此基础上逐步补充完整的通信流程。

```c++
constexpr int32_t kDeviceId = 0;
const AscendString kLocalEngine = "10.10.10.1:26000";

aclrtSetDevice(kDeviceId);

Hixl engine;
std::map<AscendString, AscendString> options;
options[OPTION_AUTO_CONNECT] = "1";

Status ret = engine.Initialize(kLocalEngine, options);
if (ret != SUCCESS) {
    printf("[ERROR] Initialize failed, ret = %u\n", ret);
    return -1;
}

// 在 Initialize 与 Finalize 之间执行内存注册、建链、数据传输等操作
// 具体方法将在后续章节介绍

engine.Finalize();
aclrtResetDevice(kDeviceId);
```

## 课后练习

本节介绍了 Initialize 和 Finalize 的设计原理、参数语义和释放顺序，请根据学习内容完成以下题目进行自测。

1. （判断题）Initialize 返回 SUCCESS 后，engine 会自动监听端口，无论 local_engine 是否包含端口。

2. （判断题）local_engine 写成 `10.10.10.1:26000` 表示当前 HIXL 作为 Server 端监听端口 26000。

3. （单选题）Initialize 失败返回 `PARAM_INVALID` 时，最常见的原因是？
   A. 远端 engine 未启动
   B. engine 字符串格式错误或 option 值非法
   C. 网络不通
   D. 设备未初始化

4. （单选题）以下哪种释放顺序是正确的？
   A. Finalize -> Disconnect -> DeregisterMem -> aclrtFree
   B. aclrtFree -> DeregisterMem -> Disconnect -> Finalize
   C. Disconnect -> DeregisterMem -> aclrtFree -> Finalize
   D. Disconnect -> Finalize -> DeregisterMem -> aclrtFree

5. （多选题）调用 Finalize 前应确认哪些条件？
   A. 不再提交新的传输请求
   B. 所有同步传输已返回，异步请求已查询到完成或失败
   C. 已完成 Disconnect 断链
   D. 已完成 DeregisterMem 并释放底层内存

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/02.02_answer.txt